In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          "That's "
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
             

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, thanks for letting me know. That's no problem at all. What time works best for you to reschedule? Best, Seán."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, thanks for letting me know. That\'s no problem at all. What time works best for you to reschedule? Best, Seán."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='5c883933afd340f6dd8d1f371d428da6')]


In [7]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, thanks for letting me know. That's no problem at all. What time works best for you to reschedule? Best, Seán.


## Approve

In [8]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1174737d-a5db-4184-ac25-598c467c6871'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"email_address": "me@example.com"}'}, '__gemini_function_call_thought_signatures__': {'call_4202333256752182903': 'EnMKcQFpFH0Tc6ykzBupAnvZBgolMLh7On4KwHy8CHBtEboOkop57U5fpy08u7KbBMriCLLIxz1wy/dzF/UT+R77rhlULFWiNYhcFWqQsYfWJd4HPMOgR7JRMT5znFjcJanMzfJS+X6a6SaVqXhHVX/9+AJr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfff-f5d3-7ac0-8499-ef03cc4e6121-0', tool_calls=[{'name': 'read_email', 'args': {'email_address': 'me@example.com'}, 'id': 'ca

## Reject

In [9]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1174737d-a5db-4184-ac25-598c467c6871'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"email_address": "me@example.com"}'}, '__gemini_function_call_thought_signatures__': {'call_4202333256752182903': 'EnMKcQFpFH0Tc6ykzBupAnvZBgolMLh7On4KwHy8CHBtEboOkop57U5fpy08u7KbBMriCLLIxz1wy/dzF/UT+R77rhlULFWiNYhcFWqQsYfWJd4HPMOgR7JRMT5znFjcJanMzfJS+X6a6SaVqXhHVX/9+AJr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfff-f5d3-7ac0-8499-ef03cc4e6121-0', tool_calls=[{'name': 'read_email', 'args': {'email_address': 'me@example.com'}, 'id': 'ca

In [13]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [11]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1174737d-a5db-4184-ac25-598c467c6871'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"email_address": "me@example.com"}'}, '__gemini_function_call_thought_signatures__': {'call_4202333256752182903': 'EnMKcQFpFH0Tc6ykzBupAnvZBgolMLh7On4KwHy8CHBtEboOkop57U5fpy08u7KbBMriCLLIxz1wy/dzF/UT+R77rhlULFWiNYhcFWqQsYfWJd4HPMOgR7JRMT5znFjcJanMzfJS+X6a6SaVqXhHVX/9+AJr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfff-f5d3-7ac0-8499-ef03cc4e6121-0', tool_calls=[{'name': 'read_email', 'args': {'email_address': 'me@example.com'}, 'id': 'ca